In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load CSV
df = pd.read_csv(r"C:\Users\sneha\Downloads\manually_labeled_chunks.csv")
assert 'chunk_text' in df.columns and 'label' in df.columns

# Train/test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['chunk_text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Datasets
train_dataset = TextDataset(train_texts, train_labels)
val_dataset = TextDataset(val_texts, val_labels)

# Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Training arguments — MINIMAL version
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
trainer.save_model("./results")
tokenizer.save_pretrained("./results")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


('./results\\tokenizer_config.json',
 './results\\special_tokens_map.json',
 './results\\vocab.txt',
 './results\\added_tokens.json',
 './results\\tokenizer.json')

In [2]:
unlabeled_df = pd.read_csv(r"C:\Users\sneha\Downloads\wiki_hybrid_chunks_300.csv")
texts = unlabeled_df['chunk_text'].tolist()


In [3]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline


In [4]:
#Load the fine-tuned model and tokenizer
model = BertForSequenceClassification.from_pretrained("./results")  # or "bert-finetuned" if saved to Hugging Face
tokenizer = BertTokenizer.from_pretrained("./results")

In [5]:
# Create classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=False)


Device set to use cpu
C:\Users\sneha\AppData\Roaming\Python\Python312\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [6]:
# Run predictions
predictions = []
batch_size = 32

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    batch_preds = classifier(batch, truncation=True, padding=True)
    predictions.extend(batch_preds)

print("Done with predictions.")


Done with predictions.


In [7]:
# Extract labels (e.g., 'LABEL_1' → 1)
labels = [int(pred['label'].split('_')[-1]) for pred in predictions]


In [8]:
# Add predictions to DataFrame
unlabeled_df['label'] = labels  # 1 = Good, 0 = Bad

In [12]:
# Save the result
unlabeled_df.to_csv(r"C:\Users\sneha\Downloads\text_quality_classifier.csv", index=False)
print("Results saved to 'text_quality_classifier.csv'")


Results saved to 'text_quality_classifier.csv'
